# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from anthropic import Anthropic
from openai import OpenAI

In [2]:
# constants

MODEL_ANTHROPIC = 'claude-opus-4-7'
MODEL_LLAMA = ['gpt-oss', 'llama3.2', 'mistral-nemo']
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get Anthropic to answer, with streaming

response = ""
display_handle = display(Markdown(""), display_id=True)
client = Anthropic(api_key=api_key)
with client.messages.stream(
    max_tokens=1024,
    model=MODEL_ANTHROPIC,
    messages=[{"role": "user", "content": question}]
) as stream:
    for text in stream.text_stream:
        response += text or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

# Code Explanation

```python
yield from {book.get("author") for book in books if book.get("author")}
```

This line combines several Python features. Let me break it down piece by piece.

## Breakdown

### 1. The set comprehension (innermost part)
```python
{book.get("author") for book in books if book.get("author")}
```
- Iterates over `books` (presumably a list of dictionaries)
- Calls `book.get("author")` to safely retrieve the author (returns `None` if missing instead of raising `KeyError`)
- The `if book.get("author")` filter excludes books where the author is missing, `None`, or an empty string (any falsy value)
- The `{...}` braces make it a **set comprehension**, which automatically removes duplicates

### 2. `yield from`
- Used inside a generator function
- Yields each element of the iterable (the set) one at a time to the caller
- Equivalent to `for author in {...}: yield author`

## Why use this pattern?

**Purpose:** Produce a stream of unique, non-empty author names from a collection of books.

**Benefits:**
| Feature | Reason |
|---|---|
| `.get("author")` | Safe access — won't crash if `"author"` key is missing |
| `if book.get("author")` | Skips books with missing/empty authors |
| `{...}` (set) | Deduplicates — each author yielded only once |
| `yield from` | Cleanly delegates iteration in a generator |

## Example

```python
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

books = [
    {"title": "A", "author": "Alice"},
    {"title": "B", "author": "Bob"},
    {"title": "C", "author": "Alice"},   # duplicate author
    {"title": "D"},                       # no author key
    {"title": "E", "author": ""},         # empty author
]

print(list(unique_authors(books)))
# Output: ['Alice', 'Bob']  (order may vary — sets are unordered)
```

## ⚠️ Caveat

Because sets are **unordered**, the order of yielded authors is not guaranteed. If you need to preserve insertion order while still deduplicating, use a `dict` instead:

```python
yield from dict.fromkeys(
    book["author"] for book in books if book.get("author")
)
```

In [8]:
# Get Llama 3.2 to answer, with streaming

response = ""
display_handle = display("", display_id=True)
for model in MODEL_LLAMA:
    response += f"### Model {model}<hr>"
    update_display(Markdown(response), display_id=display_handle.display_id)
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    stream = ollama.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        stream=True
    )    
    
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    response += '<br><hr><br>'
    update_display(Markdown(response), display_id=display_handle.display_id)


# Model gpt-oss<hr>### Short answer

`yield from {book.get("author") for book in books if book.get("author")}`

* **What it does** –  
  In a generator function it **yields every unique, truthy value of the
  `"author"` key from every dictionary in the `books` iterable**.  
  In other words, it produces a list of distinct author names, skipping any
  `None`, empty string (`""`), `0`, etc.

* **Why it is written that way** –  
  * `set` comprehension (`{…}`) builds a **set** of authors, automatically
    discarding duplicates.  
  * The `if book.get("author")` clause filters out any books that don’t
    contain a valid author or where the author value is falsy.  
  * `yield from` delegates the iteration to the set, emitting one author at
    a time.  Using `yield from` here is a concise replacement for

  ```python
  for author in {book.get("author") for book in books if book.get("author")}:
      yield author
  ```

---

## In‑depth walk‑through

| Piece | What it is | How it behaves |
|-------|------------|----------------|
| `book.get("author")` | Method on a dict that returns the value for `"author"` or `None` if the key is missing. | Provides the author name (or `None`). |
| `if book.get("author")` | Conditional filter inside the comprehension. | Only keeps truthy values (`None`, `""`, `0`, `False`, etc. are filtered out). |
| `{ ... for book in books if book.get("author")}` | **Set comprehension**. | Collects the author names into a set, eliminating duplicates and giving you an unordered collection. |
| `yield from <iterable>` | Generator delegation in Python 3.3+. | Takes any iterable (including the set) and yields each of its elements sequentially to the caller of the generator. |

### Why **set** and not a list?

* Duplicate authors would otherwise appear multiple times.  
  Example:

  ```python
  books = [
      {"author": "Alice", "title": "X"},
      {"author": "Alice", "title": "Y"},
      {"author": "Bob",   "title": "Z"}
  ]
  ```

  With a list comprehension you’d get `["Alice", "Alice", "Bob"]`; a set
  gives `{"Alice", "Bob"}`.

* Once the set is built, `yield from` simply loops over it – no
  extra bookkeeping needed.

### Why **yield from** instead of a loop?

If you wrote:

```python
authors = {book.get("author") for book in books if book.get("author")}
yield authors           # ❌ yields *the set*, not its items
```

the caller would receive a single set object.  
To stream each author separately you could write:

```python
for author in authors:
    yield author
```

Using `yield from` is the idiomatic, concise way to do exactly that:

```python
yield from authors
```

It:

1. Calls `iter(authors)` internally.
2. Yields each value one at a time.
3. Propagates any `StopIteration` automatically.

---

## Practical example

```python
def unique_authors(books):
    """Yield each distinct, non‑empty author from the books sequence."""
    yield from {book.get("author") for book in books if book.get("author")}
```

```python
>>> books = [
...     {"author": "Alice", "title": "Book A"},
...     {"author": "Bob",   "title": "Book B"},
...     {"author": "",      "title": "Book C"},      # ignored
...     {"title": "Book D"},                          # ignored
...     {"author": "Alice", "title": "Book E"}        # duplicate
... ]

>>> list(unique_authors(books))
['Bob', 'Alice']     # order is arbitrary, set is unordered
```

---

## Key Takeaways

* **`yield from`** delegates iteration – it’s cleaner and faster than a manual loop when you only want to pass through another iterable.  
* Using a **set comprehension** guarantees uniqueness and performs the filtering in a single, idiomatic statement.  
* The `if book.get("author")` guard ensures that only legitimate, truthy author names are yielded.

Thus the snippet is a compact, efficient way to expose all unique authors from a collection of book dictionaries, perfect for use inside a generator pipeline.<br><hr><br># Model llama3.2<hr>This line of Python code is using a feature called "generators" to extract the authors from a list of dictionaries. Let's break it down:

1. `{...}`: This is an expression that evaluates to an iterator, similar to a set or a dictionary.
2.`{book.get("author") for book in books if book.get("author")}` : Inside this expression, you'll find a few key concepts:
   * `for book in books`: This is an iteration over the items in the `books` list (which should be an iterable).
   * `if book.get("author")`: This condition filters out books that do not have an "author" key.
   * `book.get("author")` : For each book, this expression retrieves the value of the "author" key from the dictionary.
3. `yield from ...`: The `yield from` statement tells Python to:

    - Iterate over the result of the generator expression (which is another iterable).
    - Yield a value from that iterable for each iteration of the parent code block.
4. `yield from {...}`: It essentially flattens an iterable into separate values.

So, putting it all together: this line of Python does one or more of the following things:
  * Returns a generator producing pairs (the author name, book) for each value in your books list.
  * Returns a generator with items that are either
      * Each "book" object dictionary with an associated author.
  * The set created from those books.

However you use it:

```python
for book, author in {book.get("author") for book in books if book.get("author")} yield:
    print(f'Verifying: Author is: {author}')


def main():
      # Sample data.
      books = [
          {"title": "A Story of Mankind", 'price': 23.33, 'genre': "Thriller"},
          {"id": 7,"title": "The Story", 'price': 15.0, "genres":["Fantasy"]},
          {"author": 'Siddhal', "title": "Man of Truth", 'price': 12.01, "id": 2}
      ]
```

However this code only does so using just the books with an existing `book["author"]` key<br><hr><br># Model mistral-nemo<hr>This Python code snippet uses generator expressions with the `yield from` keyword to achieve a specific goal. Let's break it down step by step:

1. **Generator expression**: The innermost part (`{book.get("author") for book in books if book.get("author")}`) is a generator expression that generates values lazily, i.e., it produces items one at a time as needed and does not store all values in memory at once.

   - `books` is presumed to be an iterable containing dictionaries.
   - The condition `if book.get("author")` checks if the dictionary (`book`) contains the key "author".
   - If the condition holds true, it runs expression `book.get("author")`, which retrieves and yields the value associated with the key "author".

2. **yield from**: This keyword is used to delegate the iteration to another generator or iterable. In this case, it's delegating to the generator created by the generator expression.

So, in essence, this code snippet will iterate over each book in the `books` collection, check if it has an "author" key, and if true, yield that author's name one at a time. The yielded values can then be iterated over or consumed directly as they are produced by `yield from`.

Here's an example of how you might use this:

```python
books = [
    {"title": "The Catcher in the Rye", "author": "J.D. Salinger"},
    {"title": "To Kill a Mockingbird", "author": "Harper Lee"},
]

for author in (yield from {book.get("author") for book in books if book.get("author")}):
    print(author)
```

This will output:

```
J.D. Salinger
Harper Lee
```<br><hr><br>